# LeetCode #138 — Copy List with Random Pointer: map cũ → mới, và mẹo đan xen O(1)

**Mức độ:** Medium · **Chủ đề:** Hash Table, Linked List

Skeleton trên LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, x: int, next: 'Node' = None, random: 'Node' = None):
        self.val = int(x)
        self.next = next
        self.random = random
"""
class Solution:
    def copyRandomList(self, head: 'Optional[Node]') -> 'Optional[Node]':
```

Mục tiêu của notebook:

1. Hiểu vì sao sao chép `next` thì dễ, còn `random` thì khó (nó có thể trỏ tới nút **chưa tạo**).
2. Cách 1: **hai lượt** với hash map `old → new` (`O(n)` bộ nhớ).
3. Cách 2: **đan xen** bản sao ngay sau bản gốc (`A → A' → B → B' …`), rồi tách — `O(1)` bộ nhớ phụ.
4. Kiểm tra bản sao **không** trỏ về nút gốc nào và danh sách gốc được khôi phục.

# 1. Bài toán là gì?

Danh sách liên kết độ dài `n`, mỗi nút có thêm con trỏ `random` trỏ tới **bất kỳ** nút nào trong danh sách hoặc `None`. Tạo **bản sao sâu**: `n` nút mới, `next` và `random` của bản sao trỏ tới các nút **mới** tương ứng; không con trỏ nào trỏ về danh sách gốc.

LeetCode biểu diễn mỗi nút là `[val, random_index]` (`random_index` là chỉ số nút mà `random` trỏ tới, hoặc `null`).

**Constraints (theo đề):**

- `0 <= n <= 1000`
- `-10^4 <= Node.val <= 10^4`
- `Node.random` là `null` hoặc trỏ tới một nút trong danh sách.

---

## Ví dụ

```text
[[7, null], [13, 0], [11, 4], [10, 2], [1, 0]]  ->  giống hệt (về cấu trúc)
[[1, 1], [2, 1]]                                 ->  [[1, 1], [2, 1]]
[[3, null], [3, 0], [3, null]]                   ->  [[3, null], [3, 0], [3, null]]
```

Ví dụ 3 cho thấy **giá trị có thể trùng** → không thể dùng `val` làm khóa.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [6]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [7]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 465.03 ms | trên set: 0.081 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [8]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [9]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Áp dụng vào bài này

Sao chép **danh sách liên kết** có thêm con trỏ `random` trỏ tùy ý. **Hash map** `nút cũ → nút mới` cho phép lượt thứ hai nối `next` và `random` của bản sao. Cách `O(1)` bộ nhớ: **đan xen** bản sao ngay sau mỗi nút cũ, rồi tách ra.

# 3. Nền tảng

In [10]:
from typing import List, Optional
import random


class Node:
    def __init__(self, x: int, next: 'Node' = None, random: 'Node' = None):
        self.val = int(x)
        self.next = next
        self.random = random


def build(spec):
    if not spec:
        return None
    nodes = [Node(v) for v, _ in spec]
    for i, (_, r) in enumerate(spec):
        if i + 1 < len(nodes):
            nodes[i].next = nodes[i + 1]
        nodes[i].random = nodes[r] if r is not None else None
    return nodes[0]


def to_spec(head):
    nodes, node = [], head
    while node:
        nodes.append(node)
        node = node.next
    index = {id(n): i for i, n in enumerate(nodes)}
    return [[n.val, index[id(n.random)] if n.random else None] for n in nodes]


def node_ids(head):
    out, node = set(), head
    while node:
        out.add(id(node))
        node = node.next
    return out

EX = [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
print(to_spec(build(EX)))

[[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]


`to_spec` đọc lại cấu trúc theo **chỉ số** — nếu `random` trỏ ra ngoài danh sách đang đọc (ví dụ trỏ về danh sách gốc), `index[id(...)]` sẽ báo lỗi. Đó là một kiểm tra tự nhiên cho bản sao sâu.

# 4. Vì sao khó?

Duyệt một lượt, tạo nút mới cho mỗi nút cũ: `next` thì nối được ngay. Nhưng `random` của nút thứ `i` có thể trỏ tới nút thứ `j > i` — **chưa được tạo**. Và kể cả `j < i`, ta cần biết **bản sao** của nút `j` là đối tượng nào.

⇒ Cần một cách tra "nút cũ này ứng với nút mới nào".

# 5. Cách 1: Hash map, hai lượt

1. Lượt 1: với mỗi nút cũ, tạo nút mới, lưu `mp[old] = new`.
2. Lượt 2: `mp[old].next = mp.get(old.next)`, `mp[old].random = mp.get(old.random)`.

In [11]:
def copy_two_pass(head: Optional[Node]) -> Optional[Node]:
    mp = {None: None}
    node = head
    while node:
        mp[node] = Node(node.val)
        node = node.next
    node = head
    while node:
        mp[node].next = mp[node.next]
        mp[node].random = mp[node.random]
        node = node.next
    return mp[head]

h = build(EX); c = copy_two_pass(h)
print(to_spec(c), "| chung nút với gốc?", bool(node_ids(h) & node_ids(c)))

[[7, None], [13, 0], [11, 4], [10, 2], [1, 0]] | chung nút với gốc? False


Mẹo `mp = {None: None}`: tra `mp[None]` trả `None` — khỏi phải viết `if old.random else None`.

Khóa của map là **chính đối tượng nút** (hash theo `id`), nên giá trị trùng không gây vấn đề.

# 6. Cách 2: Đan xen — `O(1)` bộ nhớ phụ

Thay vì map, đặt bản sao **ngay sau** nút gốc: `A → A' → B → B' → C → C'`. Khi đó "bản sao của `X`" chính là `X.next`.

1. **Đan:** với mỗi `X`, chèn `X' = Node(X.val)` vào giữa `X` và `X.next`.
2. **Gán random:** `X'.random = X.random.next` (bản sao của `X.random`), nếu `X.random` không rỗng.
3. **Tách:** khôi phục `X.next = X'.next` và nối `X'.next = X'.next.next`.

```text
gốc:     A -> B -> C
đan:     A -> A' -> B -> B' -> C -> C'
random:  A'.random = A.random.next
tách:    A -> B -> C       A' -> B' -> C'
```

In [12]:
def copy_interleave(head: Optional[Node]) -> Optional[Node]:
    if not head:
        return None
    node = head
    while node:                                   # 1. đan
        node.next = Node(node.val, node.next)
        node = node.next.next
    node = head
    while node:                                   # 2. random
        if node.random:
            node.next.random = node.random.next
        node = node.next.next
    new_head = head.next
    node = head
    while node:                                   # 3. tách
        copy = node.next
        node.next = copy.next
        copy.next = copy.next.next if copy.next else None
        node = node.next
    return new_head

h = build(EX); c = copy_interleave(h)
print("bản sao:", to_spec(c), "| gốc được khôi phục:", to_spec(h) == EX, "| chung nút?", bool(node_ids(h) & node_ids(c)))

bản sao: [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]] | gốc được khôi phục: True | chung nút? False


# 7. Mô phỏng cách đan xen

In [13]:
def show_chain(head, tag=lambda n: str(n.val)):
    out, node = [], head
    while node:
        out.append(tag(node))
        node = node.next
    return " -> ".join(out)

def trace_interleave(spec):
    head = build(spec)
    originals = set()
    node = head
    while node:
        originals.add(id(node)); node = node.next
    tag = lambda n: f"{n.val}" if id(n) in originals else f"{n.val}'"
    print("Gốc   :", show_chain(head, tag))
    node = head
    while node:
        node.next = Node(node.val, node.next)
        node = node.next.next
    print("Đan   :", show_chain(head, tag))
    node = head
    while node:
        if node.random:
            node.next.random = node.random.next
            print(f"   {node.val}'.random = ({node.val}.random = {node.random.val}).next = {tag(node.random.next)}")
        node = node.next.next
    new_head = head.next
    node = head
    while node:
        copy = node.next
        node.next = copy.next
        copy.next = copy.next.next if copy.next else None
        node = node.next
    print("Tách  : gốc =", show_chain(head, tag), "| bản sao =", show_chain(new_head, tag))
    print("spec bản sao:", to_spec(new_head))
    return new_head

_ = trace_interleave([[1, 1], [2, 1], [3, 0]])

Gốc   : 1 -> 2 -> 3
Đan   : 1 -> 1' -> 2 -> 2' -> 3 -> 3'
   1'.random = (1.random = 2).next = 2'
   2'.random = (2.random = 2).next = 2'
   3'.random = (3.random = 1).next = 1'
Tách  : gốc = 1 -> 2 -> 3 | bản sao = 1' -> 2' -> 3'
spec bản sao: [[1, 1], [2, 1], [3, 0]]


# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def copyRandomList(self, head: 'Optional[Node]') -> 'Optional[Node]':
        if not head:
            return None
        node = head
        while node:
            node.next = Node(node.val, node.next)
            node = node.next.next
        node = head
        while node:
            if node.random:
                node.next.random = node.random.next
            node = node.next.next
        new_head = head.next
        node = head
        while node:
            copy = node.next
            node.next = copy.next
            copy.next = copy.next.next if copy.next else None
            node = node.next
        return new_head

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


def verify(f, spec):
    h = build(spec)
    c = f(h)
    assert to_spec(h) == spec, "danh sách gốc bị thay đổi"
    assert not (node_ids(h) & node_ids(c)), "bản sao dùng chung nút với gốc"
    return to_spec(c)


sol = Solution()
check("Ví dụ 1", verify(sol.copyRandomList, EX), EX)
check("Ví dụ 2", verify(sol.copyRandomList, [[1, 1], [2, 1]]), [[1, 1], [2, 1]])
check("Ví dụ 3", verify(sol.copyRandomList, [[3, None], [3, 0], [3, None]]), [[3, None], [3, 0], [3, None]])
check("Rỗng", sol.copyRandomList(None), None)
check("Tự trỏ", verify(sol.copyRandomList, [[5, 0]]), [[5, 0]])

random.seed(138)
for _ in range(3000):
    n = random.randint(0, 12)
    spec = [[random.randint(-3, 3), random.choice([None] + list(range(n)))] for _ in range(n)]
    for f in (sol.copyRandomList, copy_two_pass):
        assert verify(f, spec) == spec
print("3000 danh sách ngẫu nhiên: bản sao đúng, độc lập, gốc được khôi phục")

Ví dụ 1      PASS  [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
Ví dụ 2      PASS  [[1, 1], [2, 1]]
Ví dụ 3      PASS  [[3, None], [3, 0], [3, None]]
Rỗng         PASS  None
Tự trỏ       PASS  [[5, 0]]


3000 danh sách ngẫu nhiên: bản sao đúng, độc lập, gốc được khôi phục


# 9. Vì sao đúng? Độ phức tạp?

**Đan xen:** sau bước 1, với mọi nút gốc `X`, `X.next` là bản sao `X'` và `X'.next` là nút gốc kế tiếp. Bước 2: `X.random` (nếu có) là một nút gốc `Y`, nên `Y.next = Y'` — đúng là bản sao cần trỏ tới. Bước 3 tách xen kẽ: các nút ở vị trí chẵn (gốc) nối lại với nhau, vị trí lẻ (bản sao) nối với nhau.

```text
             Time   Bộ nhớ phụ
Hash map     O(n)   O(n)
Đan xen      O(n)   O(1)   (không tính các nút bản sao — đó là output)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Gán `random` trỏ về nút **gốc**

In [16]:
def wrong_random_old(head):
    mp = {None: None}
    node = head
    while node:
        mp[node] = Node(node.val); node = node.next
    node = head
    while node:
        mp[node].next = mp[node.next]
        mp[node].random = node.random              # SAI: trỏ về nút gốc
        node = node.next
    return mp[head]

h = build(EX); c = wrong_random_old(h)
try:
    to_spec(c)
except KeyError:
    print("Sai : random của bản sao trỏ ra ngoài bản sao (về danh sách gốc)")

Sai : random của bản sao trỏ ra ngoài bản sao (về danh sách gốc)


## Lỗi 2: Dùng `val` làm khóa

In [17]:
def wrong_key_by_val(head):
    mp = {}
    node = head
    while node:
        mp[node.val] = Node(node.val)              # SAI: val có thể trùng
        node = node.next
    return mp

h = build([[3, None], [3, 0], [3, None]])
print("Cần 3 nút bản sao, map theo val chỉ tạo được:", len(wrong_key_by_val(h)))

Cần 3 nút bản sao, map theo val chỉ tạo được: 1


Ba nút có cùng `val = 3` bị gộp vào **một** khóa — bản sao chỉ còn một nút; nối `next` theo map này sẽ tạo vòng tự trỏ. Khóa phải là **chính đối tượng nút**.

## Lỗi 3: Đan xen nhưng quên khôi phục danh sách gốc

LeetCode kiểm tra cả danh sách gốc — nếu chỉ lấy bản sao mà để gốc ở trạng thái đan xen, sẽ bị chấm sai.

In [18]:
def wrong_no_restore(head):
    node = head
    while node:
        node.next = Node(node.val, node.next); node = node.next.next
    node = head
    while node:
        if node.random:
            node.next.random = node.random.next
        node = node.next.next
    new_head = head.next
    node = new_head
    while node and node.next:                      # SAI: chỉ nối bản sao, không sửa gốc
        node.next = node.next.next
        node = node.next
    return new_head

h = build([[1, None], [2, 0]])
c = wrong_no_restore(h)
print("Bản sao:", to_spec(c), "| gốc sau đó:", show_chain(h))

Bản sao: [[1, None], [2, 0]] | gốc sau đó: 1 -> 1 -> 2


Danh sách gốc giờ là `1 → 1 → 2` (nút gốc `1` vẫn trỏ vào bản sao `1'`).

## Lỗi 4: Tách sai khi bản sao cuối (`copy.next` là `None`)

Viết `copy.next = copy.next.next` mà không kiểm tra → `AttributeError` ở nút cuối.

# 11. Mẹo nhận diện pattern

- **Sao chép sâu cấu trúc có con trỏ "bất kỳ"** → map `old → new` (giống #133 Clone Graph).
- **Cần `O(1)` bộ nhớ trên danh sách liên kết** → mượn chính cấu trúc danh sách: đan xen, đảo tạm, hay đánh dấu bằng con trỏ — rồi **khôi phục**.

Bài liên quan: **#133 Clone Graph**, **#1485 Clone Binary Tree With Random Pointer**, **#1490 Clone N-ary Tree**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[[1, 2], [2, None], [3, 1]]`, vẽ danh sách sau bước đan và cho biết `random` của từng bản sao.

## Bài 2 — Điền chỗ trống (hash map)

In [19]:
def exercise_2_copy_random_list(head: Optional[Node]) -> Optional[Node]:
    mp = {None: None}
    # TODO: lượt 1 tạo nút mới; lượt 2 gán next và random qua mp
    return mp.get(head)


# assert verify(exercise_2_copy_random_list, [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]) == [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
# assert exercise_2_copy_random_list(None) is None

## Bài 3 — Biến thể: sao chép một lượt bằng đệ quy + map

Viết `copy(node)`: nếu đã có trong map thì trả; ngược lại tạo nút, **ghi map trước**, rồi `copy.next = copy(node.next)`, `copy.random = copy(node.random)`. (Giống #133.)

In [20]:
def exercise_3_copy_recursive(head: Optional[Node]) -> Optional[Node]:
    # TODO
    pass


# assert verify(exercise_3_copy_recursive, [[1, 1], [2, 1]]) == [[1, 1], [2, 1]]
# assert verify(exercise_3_copy_recursive, [[5, 0]]) == [[5, 0]]

## Bài 4 — Biến thể: kiểm tra một danh sách có phải bản sao sâu hợp lệ không

Viết `is_valid_copy(orig, cp)`: cùng cấu trúc `to_spec`, và không có nút chung. Phải phát hiện được bản sao có `random` trỏ về danh sách gốc.

In [21]:
def exercise_4_is_valid_copy(orig: Optional[Node], cp: Optional[Node]) -> bool:
    # TODO
    pass


# h = build([[7, None], [13, 0]]); assert exercise_4_is_valid_copy(h, copy_two_pass(h)) is True
# h = build([[7, None], [13, 0]]); assert exercise_4_is_valid_copy(h, wrong_random_old(h)) is False
# h = build([[7, None], [13, 0]]); assert exercise_4_is_valid_copy(h, h) is False

## Bài 5 — Mini challenge: đếm số nút có `random` trỏ **về phía trước** trong `O(1)` bộ nhớ

"Về phía trước" nghĩa là `random` trỏ tới nút có chỉ số **lớn hơn hoặc bằng** chính nó. Không dùng map/set; được phép sửa tạm danh sách nhưng phải khôi phục.

Gợi ý nhanh: đánh số tạm thời bằng cách đan nút "chỉ số" vào sau mỗi nút (như bước đan), hoặc lưu chỉ số vào `val` tạm thời rồi trả lại.

In [22]:
def exercise_5_count_forward_random(head: Optional[Node]) -> int:
    # TODO
    pass


# h = build([[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]); assert exercise_5_count_forward_random(h) == 1 and to_spec(h) == [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
# h = build([[5, 0]]); assert exercise_5_count_forward_random(h) == 1

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_interleave` sau khi tự làm.

**Bài 3**
- Mức 1: `mp = {}`; hàm `copy(node)`: `None` → `None`.
- Mức 2: với `n = 1000`, đệ quy theo `next` sâu 1000 tầng — gần giới hạn của Python; có thể tăng `sys.setrecursionlimit`.

**Bài 4**
- Mức 1: kết hợp `node_ids` và `to_spec` (bắt `KeyError` → `False`).

**Bài 5**
- Mức 1: đan một nút "chỉ số" `Node(i)` ngay sau mỗi nút gốc — không cần map hay list.
- Mức 2: khi đã đan, chỉ số của `X` là `X.next.val`; so `X.random.next.val >= X.next.val`. Cuối cùng tách bỏ các nút chỉ số.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
_ = trace_interleave([[1, 2], [2, None], [3, 1]])

Gốc   : 1 -> 2 -> 3
Đan   : 1 -> 1' -> 2 -> 2' -> 3 -> 3'
   1'.random = (1.random = 3).next = 3'
   3'.random = (3.random = 2).next = 2'
Tách  : gốc = 1 -> 2 -> 3 | bản sao = 1' -> 2' -> 3'
spec bản sao: [[1, 2], [2, None], [3, 1]]


## Đáp án Bài 2

In [24]:
def answer_2_copy_random_list(head: Optional[Node]) -> Optional[Node]:
    return copy_two_pass(head)

assert verify(answer_2_copy_random_list, [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]) == [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
assert answer_2_copy_random_list(None) is None
print("Bài 2 PASS (giống copy_two_pass ở mục 5)")

Bài 2 PASS (giống copy_two_pass ở mục 5)


## Đáp án Bài 3

In [25]:
def answer_3_copy_recursive(head: Optional[Node]) -> Optional[Node]:
    mp = {}
    def copy(node):
        if node is None:
            return None
        if node in mp:
            return mp[node]
        cp = Node(node.val)
        mp[node] = cp
        cp.next = copy(node.next)
        cp.random = copy(node.random)
        return cp
    return copy(head)

assert verify(answer_3_copy_recursive, [[1, 1], [2, 1]]) == [[1, 1], [2, 1]]
assert verify(answer_3_copy_recursive, [[5, 0]]) == [[5, 0]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_is_valid_copy(orig: Optional[Node], cp: Optional[Node]) -> bool:
    if node_ids(orig) & node_ids(cp):
        return False
    try:
        return to_spec(orig) == to_spec(cp)
    except KeyError:
        return False

h = build([[7, None], [13, 0]]); assert answer_4_is_valid_copy(h, copy_two_pass(h)) is True
h = build([[7, None], [13, 0]]); assert answer_4_is_valid_copy(h, wrong_random_old(h)) is False
h = build([[7, None], [13, 0]]); assert answer_4_is_valid_copy(h, h) is False
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_count_forward_random(head: Optional[Node]) -> int:
    node, i = head, 0
    while node:                                    # đan nút chỉ số
        node.next = Node(i, node.next)
        node = node.next.next
        i += 1
    count, node = 0, head
    while node:
        if node.random and node.random.next.val >= node.next.val:
            count += 1
        node = node.next.next
    node = head
    while node:                                    # gỡ nút chỉ số
        node.next = node.next.next
        node = node.next
    return count

h = build([[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]); assert answer_5_count_forward_random(h) == 1 and to_spec(h) == [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
h = build([[5, 0]]); assert answer_5_count_forward_random(h) == 1
for _ in range(2000):
    n = random.randint(0, 10)
    spec = [[random.randint(-3, 3), random.choice([None] + list(range(n)))] for _ in range(n)]
    h = build(spec)
    assert answer_5_count_forward_random(h) == sum(1 for i, (_, r) in enumerate(spec) if r is not None and r >= i)
    assert to_spec(h) == spec
print("Bài 5 PASS (và danh sách được khôi phục)")

Bài 5 PASS (và danh sách được khôi phục)


# 15. Tổng kết

> Cần biết "bản sao của nút X là gì". Map `old → new` (hai lượt) là cách thẳng; đan xen `X → X' → …` biến "bản sao của X" thành `X.next`, gán `X'.random = X.random.next`, rồi tách và **khôi phục** gốc — `O(1)` bộ nhớ phụ.

---

## Checklist trước khi nộp

- [ ] `random` của bản sao trỏ tới **bản sao** (không phải nút gốc)?
- [ ] Khóa map là **nút**, không phải `val`?
- [ ] Đan xen: khôi phục danh sách gốc?
- [ ] Xử lý `head = None` và nút cuối khi tách?